In [1]:
from keras.applications import vgg16

img_rows = 160
img_cols = 160 

model =   vgg16.VGG16(weights = 'imagenet', 
                     include_top = False, 
                     input_shape = (img_rows, img_cols, 3))

I0000 00:00:1790939364.266841    8635 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790939364.267417    8635 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790939364.407205    8635 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790939366.793285    8635 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
for layer in model.layers:
    layer.trainable = False
 
for (i,layer) in enumerate(model.layers):
    print(str(i) + " "+ layer.__class__.__name__, layer.trainable)

0 InputLayer False
1 Conv2D False
2 Conv2D False
3 MaxPooling2D False
4 Conv2D False
5 Conv2D False
6 MaxPooling2D False
7 Conv2D False
8 Conv2D False
9 Conv2D False
10 MaxPooling2D False
11 Conv2D False
12 Conv2D False
13 Conv2D False
14 MaxPooling2D False
15 Conv2D False
16 Conv2D False
17 Conv2D False
18 MaxPooling2D False


In [3]:
def addTopModel(bottom_model, num_classes, D=256):
    """creates the top or head of the model that will be 
    placed ontop of the bottom layers"""
    top_model = bottom_model.output
    # top_model = Flatten(name = "flatten")(top_model)
    top_model = GlobalAveragePooling2D()(top_model)
    top_model = Dense(D, activation = "relu")(top_model)
    top_model = Dropout(0.3)(top_model)
    top_model = Dense(num_classes, activation = "softmax")(top_model)
    return top_model

In [4]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten,GlobalAveragePooling2D
from keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D
# from keras.layers.normalization import BatchNormalization
from keras.models import Model

num_classes = 36

FC_Head = addTopModel(model, num_classes)

modelnew = Model(inputs=model.input, outputs=FC_Head)

print(modelnew.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 100, 100, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 100, 100, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 50, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 50, 50, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 50, 50, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 25, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 25, 25, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 25, 25, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 25, 25, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 12, 12, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 12, 12, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 12, 12, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 6, 6, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         9,252 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,903,844 (60.67 MB)

 Trainable params: 1,189,156 (4.54 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

None


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input




train_data_dir = "../data/fruits_data/train"
validation_data_dir = "../data/fruits_data/validation"
print("Training:", len(train_data_dir))
print("Validation:", len(validation_data_dir))

# class_names = sorted(
#     set(train_data_dir["label"])
# )

# print(class_names)
# print(len(class_names))

# # Create class -> integer mapping
# class_to_index = {
#     class_name: index
#     for index, class_name in enumerate(class_names)
# }

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
      rotation_range=20,
      width_shift_range=0.2,
      height_shift_range=0.2,
      horizontal_flip=True,
      fill_mode='nearest')
 
validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
 
train_batchsize = 16
val_batchsize = 16
 
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=train_batchsize,
    class_mode="categorical",
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=val_batchsize,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)

class_to_index = train_generator.class_indices

class_names = list(class_to_index.keys())



print()
print("========================================")
print("Dataset Information")
print("========================================")
print("Training images:", train_generator.samples)
print("Validation images:", validation_generator.samples)
print("Number of classes:", num_classes)

print()
print("Class mapping:")
print(class_to_index)
import json

CLASS_NAMES_PATH = "../ML_models/class_names.json"

with open(CLASS_NAMES_PATH, "w") as file:
    json.dump(class_names, file, indent=4)

print(f"Class names saved to: {CLASS_NAMES_PATH}")

Training: 25
Validation: 30
Found 3115 images belonging to 36 classes.
Found 351 images belonging to 36 classes.

Dataset Information
Training images: 3115
Validation images: 351
Number of classes: 36

Class mapping:
{'apple': 0, 'banana': 1, 'beetroot': 2, 'bell pepper': 3, 'cabbage': 4, 'capsicum': 5, 'carrot': 6, 'cauliflower': 7, 'chilli pepper': 8, 'corn': 9, 'cucumber': 10, 'eggplant': 11, 'garlic': 12, 'ginger': 13, 'grapes': 14, 'jalepeno': 15, 'kiwi': 16, 'lemon': 17, 'lettuce': 18, 'mango': 19, 'onion': 20, 'orange': 21, 'paprika': 22, 'pear': 23, 'peas': 24, 'pineapple': 25, 'pomegranate': 26, 'potato': 27, 'raddish': 28, 'soy beans': 29, 'spinach': 30, 'sweetcorn': 31, 'sweetpotato': 32, 'tomato': 33, 'turnip': 34, 'watermelon': 35}
Class names saved to: ../ML_models/class_names.json


In [ ]:
modelnew.compile(loss = 'categorical_crossentropy',
              optimizer =  "adam"  ,
              metrics = ['accuracy'])

nb_train_samples = train_generator.samples
nb_validation_samples = validation_generator.samples
epochs = 50
batch_size = 16

from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "/app/ML_models/checkpoints/fruitveg_{epoch:03d}.keras",
    save_freq="epoch",
    save_best_only=False
)

history = modelnew.fit(
    train_generator,
    # steps_per_epoch = nb_train_samples // batch_size,
    epochs = epochs,
    # callbacks = callbacks,
    validation_data = validation_generator,
    # validation_steps = nb_validation_samples // batch_size)
)

modelnew.save("../ML_models/fruits_veg_identify_v3.keras")

Epoch 1/50


I0000 00:00:1790939370.776017    8635 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


  6/195 ━━━━━━━━━━━━━━━━━━━━ 1:53 601ms/step - accuracy: 0.0729 - loss: 4.0237

/usr/local/lib/python3.11/site-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


195/195 ━━━━━━━━━━━━━━━━━━━━ 147s 752ms/step - accuracy: 0.1727 - loss: 3.1199 - val_accuracy: 0.5071 - val_loss: 2.0211
Epoch 2/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 125s 641ms/step - accuracy: 0.3284 - loss: 2.3970 - val_accuracy: 0.6097 - val_loss: 1.4430
Epoch 3/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 124s 633ms/step - accuracy: 0.4064 - loss: 2.0983 - val_accuracy: 0.6895 - val_loss: 1.1970
Epoch 4/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 128s 659ms/step - accuracy: 0.4443 - loss: 1.9214 - val_accuracy: 0.7521 - val_loss: 1.0052
Epoch 5/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 121s 621ms/step - accuracy: 0.4896 - loss: 1.7789 - val_accuracy: 0.7778 - val_loss: 0.9499
Epoch 6/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 117s 602ms/step - accuracy: 0.4986 - loss: 1.6803 - val_accuracy: 0.7863 - val_loss: 0.8241
Epoch 7/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 118s 607ms/step - accuracy: 0.5066 - loss: 1.6334 - val_accuracy: 0.7949 - val_loss: 0.7876
Epoch 8/50
195/195 ━━━━━━━━━━━━━━━━━━━━ 119s 608ms/step - accuracy: 0.5300 - loss: 1.56

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image


def predict_single_image(img_path: str):
    """Load, display, and predict a single image."""

    print(f"\n--- Evaluation Request for: {img_path} ---")

    # Check if file exists
    if not os.path.exists(img_path):
        print(f"Error: Target path '{img_path}' does not exist.")
        return
    IMG_SIZE =(img_rows,img_cols)
    # Load image
    img = image.load_img(
        img_path,
        target_size=IMG_SIZE
    )

    # Display image
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # Convert image to NumPy array
    img_array = image.img_to_array(img)

    # Rescale exactly like the training generator
    img_array = preprocess_input(img_array)

    # Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)

    # Make prediction
    predictions = modelnew.predict(img_batch, verbose=0)[0]

    # Get best prediction
    best_match_idx = np.argmax(predictions)
    confidence_score = predictions[best_match_idx]

    # Get class name
    predicted_class = class_names[best_match_idx]

    print()
    print("========================================")
    print("Prediction Result")
    print("========================================")
    print(
        f"Identified as: '{predicted_class}'"
    )
    print(
        f"Confidence: {confidence_score * 100:.2f}%"
    )

    # Display top 10 predictions
    top_n = 10

    top_indices = np.argsort(predictions)[-top_n:][::-1]

    print()
    print("Top Predictions")
    print("----------------------------------------")

    for rank, idx in enumerate(top_indices, start=1):
        print(
            f"{rank}. {class_names[idx]}: "
            f"{predictions[idx] * 100:.2f}%"
        )


# Test prediction
sample_test_path = "./banana1.jpg"

predict_single_image(sample_test_path)